# Homework 1 - Intro and Data Sources

The four questions of Module 1: the S&P 500 constituent list, the world indices, the
drawdowns of the index since 1950 and the price reaction to earnings surprises.

**The outcome**

| # | Question | Answer |
|---|---|---|
| 1 | Which (full) year had the most additions to the S&P 500 since 2020? | **2025** (17 additions) |
| 2 | How many world indices beat the S&P 500 in the first eight months of 2026? | **2** (the Nikkei and the TSX) |
| 3 | What is the median drawdown of the corrections of at least 5% since 1950? | **8** (median 7.99%) |
| 4 | What is the median two-day return after a positive AMZN earnings surprise? | **0.35** (median 0.3528%) |

The cells below reproduce every number. The code lives in
`homework1/src/hw1_questions.py`; the notebook calls it, so there is one implementation
of each question and the stored outputs cannot drift away from the code.

In [1]:
# Colab: uncomment and run the next line first.
# !pip install -q pandas numpy pyarrow scikit-learn yfinance curl_cffi
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def repo_root(start: Path) -> Path:
    """Walk up the tree until the repository layout is recognised."""
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "scripts").exists():
            return path
    return start


REPO = repo_root(Path.cwd().resolve())
HW = REPO / "homework1"
sys.path.insert(0, str(HW / "src"))

print("repository :", REPO)
print("python     :", sys.version.split()[0])

repository : /run/csi/mount-root/nas/eab0d61a99b6696edb3d2aff87b585e8/repo
python     : 3.11.2


## Question 1 - S&P 500 additions per year

The constituent list on Wikipedia carries the date each company joined the index. The
year is taken from that date and the companies are counted per year.

The assignment asks for a **full** year, so the current, partial year is excluded: the
2026 list is incomplete by construction, and counting it would compare a half year
against a full one. The script reports both readings.

In [2]:
import hw1_questions

hw1_questions.question_1()

Q1 counts since 2020: {2020: 10, 2021: 10, 2022: 14, 2023: 13, 2024: 16, 2025: 17, 2026: 19}
Q1 max year incl. current partial: 2026 | best FULL year: 2025 ( 17 additions ) ties: [2025]
Q1 additional (>20y): 223 of 503


## Question 2 - World indices against the S&P 500

Every index of the Wikipedia list is downloaded with `yfinance` and its return from the
last close of 2025 to the same date in 2026 is computed, then compared with the S&P 500
over the same window. The window is the same for both, because a year-to-date
comparison against a different close would not be a comparison.

In [3]:
hw1_questions.question_2()

Q2 United States - S&P 500          2026-01-02 -> 2026-08-20     11.41%
Q2 China - Shanghai Composite       2026-01-05 -> 2026-08-20     -2.97%
Q2 Hong Kong - Hang Seng            2026-01-02 -> 2026-08-20     -2.43%
Q2 Australia - S&P/ASX 200          2026-01-02 -> 2026-08-20      4.08%
Q2 India - Nifty 50                 2026-01-01 -> 2026-08-20     -7.32%
Q2 Canada - S&P/TSX Composite       2026-01-02 -> 2026-08-20     14.06%
Q2 Germany - DAX                    2026-01-02 -> 2026-08-20      5.88%
Q2 United Kingdom - FTSE 100        2026-01-02 -> 2026-08-20      8.01%
Q2 Japan - Nikkei 225               2026-01-05 -> 2026-08-20     27.75%
Q2 Mexico - IPC Mexico              2026-01-02 -> 2026-08-20      0.32%
Q2 Brazil - Ibovespa                2026-01-02 -> 2026-08-20      4.60%
Q2 count better than S&P 500: 2 {'Canada - S&P/TSX Composite': 14.0574656152199, 'Japan - Nikkei 225': 27.750744826533214}
   sensitivity[as_written_exclusive_end (2026-01-01..2026-08-20)]: sp500=11.41% -> n_

## Question 3 - Drawdowns of the index since 1950

The index series is turned into a peak-to-trough drawdown series, and every episode
that went at least 5% below its previous peak is collected. The median of those
episodes is the answer, and the ten largest are printed so the median can be checked
against the distribution.

In [4]:
hw1_questions.question_3()

Q3 top 10 corrections (method check):
  ath_date trough_date  drawdown_pct  duration_days
2007-10-09  2009-03-09     56.775388            517
2000-03-24  2002-10-09     49.146948            929
1973-01-11  1974-10-03     48.203593            630
1968-11-29  1970-05-26     36.061641            543
2020-02-19  2020-03-23     33.924960             33
1987-08-25  1987-12-04     33.509515            101
1961-12-12  1962-06-26     27.973568            196
1980-11-28  1982-08-12     27.113582            622
2022-01-03  2022-10-12     25.425097            282
1966-02-09  1966-10-07     22.177335            240
Q3 n corrections >=5%: 74 | median drawdown: 7.986 | median duration: 40.5


## Question 4 - The price reaction to an earnings surprise

The assignment window is reconstructed explicitly: the earnings dates of the period the
homework was written in, the surprise on each of them, and the return of the stock over
the two days that follow. Only the events inside that window are used, and the median
is taken over the positive surprises, because the question asks about the reaction to a
positive surprise.

In [5]:
hw1_questions.question_4()

Q4 assignment-era events: 24 | positive: 20
Q4 median 2d return on positive surprises: 0.3528 %  -> option 0.35
Q4 mean: 0.4635 %
Q4 corr(return, surprise) all: 0.2191 | positive-only: 0.3307
earnings_date  surprise_pct  return_2d_pct
   2026-07-30        215.14      19.823514
   2026-04-29         68.18       2.063914
   2026-02-05          0.22      -9.730030
   2025-10-30         25.20       6.044289
   2025-07-31         27.19      -6.707503
   2025-05-01         16.77       3.014856
   2025-02-06         25.29      -2.972437
   2024-10-31         25.22       2.698074
   2024-08-01         23.76     -10.204301
   2024-04-30         17.67      -1.083116
   2024-02-01         24.38      10.702320
   2023-10-26         62.22       5.231072
   2023-08-03         91.19       8.860463
   2023-04-27         44.29       0.447698
   2023-02-02        -82.20      -1.673801
   2022-10-27         35.53     -10.591388
   2022-07-28       -273.30      11.556581
   2022-04-28       -190.58     -1

## The answers

In [6]:
# The answers, and the file the scripts write. The notebook and the script produce the
# same JSON, so the answer in the README, the answer in the file and the answer printed
# here cannot drift apart.
import json

answers = hw1_questions.answers
out = HW / "output" / "hw1_answers.json"
out.write_text(json.dumps(answers, indent=2, default=str))

for key, value in answers.items():
    print("=" * 70)
    print(key.upper(), "-", value.get("question", ""))
    for field, item in value.items():
        if field == "question":
            continue
        if isinstance(item, (dict, list)) and len(str(item)) > 200:
            print("  " + field + ": <" + type(item).__name__ + " of " + str(len(item)) + ">")
        else:
            print("  " + field + ": " + str(item))

print()
print("Saved:", out)

Q1 - Which year had the highest number of additions (starting from 2020)?
  counts_by_year: {2020: 10, 2021: 10, 2022: 14, 2023: 13, 2024: 16, 2025: 17, 2026: 19}
  max_year_including_current_partial: 2026
  max_year_full_years_only: 2025
  max_count_full_years_only: 17
  tied_full_years: [2025]
  additional_stocks_in_index_over_20_years: 223
  total_constituents: 503
  snapshot_date: 2026-11-04
Q2 - How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?
  sp500_ytd_pct: 11.412019253393835
  all_indexes: <dict of 11>
  windows: <dict of 11>
  n_better_than_sp500: 2
  better_than_sp500: {'Canada - S&P/TSX Composite': 14.0574656152199, 'Japan - Nikkei 225': 27.750744826533214}
  sensitivity: <dict of 3>
Q3 - Median drawdown (in %) of significant market corrections in the S&P 500 index
  data_start: 1950-01-03
  data_end: 2026-11-06
  n_ath_points: 1510
  n_corrections_total: 1509
  n_corrections_ge_5pct: 74
  median_drawdown_pct: 7.9863